<a href="https://colab.research.google.com/github/WSmithDR/espol-bootcamp-data_science/blob/optimizacion_de_hiperparametros-solucion/Ejemplos_y_Ejercicio_Hiperpar%C3%A1metros_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejemplo Hiperparámetros Random Forest
Coding Bootcamps ESPOL, Machine Learning and Predictions, Cohorte II

Instructor: Galo Castillo López


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('./millas_por_galon.csv')

In [ ]:
df.head()

* nb_cilindros: la unidad de potencia del automóvil donde la gasolina se convierte en energía
* desplazamiento: desplazamiento del motor del auto
* potencia: tasa de rendimiento del motor en caballos de fuerza
* peso: el peso de un coche
* aceleracion: la aceleración del auto
* anio: anio en el que el auto fue lanzado al mercado
* origen: el origen del coche
* modelo: el nombre del auto
* mpg: Millaje/Millas por galón

In [ ]:
df.describe()

In [ ]:
sns.pairplot(df)

In [ ]:
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
X = df[['desplazamiento',	'peso',	'aceleracion',	'anio']]

y = df['mpg']

X_train_tmp, X_test, y_train_tmp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_tmp, y_train_tmp, test_size=0.20, random_state=42)


In [ ]:
model_rl = LinearRegression()
model_rl.fit(X_train, y_train)

In [ ]:
y_pred_rl = model_rl.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_rl)
mse = mean_squared_error(y_val, y_pred_rl)
r2 = r2_score(y_val, y_pred_rl)

# Resultados
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

In [ ]:
plt.scatter(x=y_pred_rl, y=y_val, c='red', marker='o')
plt.plot([5, max(max(y_pred_rl), max(y_val))],
         [5, max(max(y_pred_rl), max(y_val))], 'k-')
plt.xlabel('Pred')
plt.ylabel('Real')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
RandomForestRegressor?

In [ ]:
model_rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                                 min_samples_split=2, max_features=0.9,
                                 random_state=0)
model_rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = model_rf.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_rf)
mse = mean_squared_error(y_val, y_pred_rf)
r2 = r2_score(y_val, y_pred_rf)

# Resultados
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

In [ ]:
plt.scatter(x=y_pred_rf, y=y_val, c='green', marker='^')
plt.plot([5, max(max(y_pred_rf), max(y_val))],
         [5, max(max(y_pred_rf), max(y_val))], 'k-')
plt.xlabel('Pred')
plt.ylabel('Real')
plt.show()

In [ ]:
plt.scatter(x=y_pred_rl, y=y_val, c='red', marker='o', label='regresión lineal')
plt.scatter(x=y_pred_rf, y=y_val, c='green', marker='^', label='random forest')
plt.plot([5, max(max(y_pred_rl), max(y_pred_rf), max(y_val))],
         [5, max(max(y_pred_rl), max(y_pred_rf), max(y_val))], 'k-')
plt.xlabel('Pred')
plt.ylabel('Real')
plt.legend()
plt.show()

**GridSearch**

In [ ]:
from sklearn.model_selection import GridSearchCV
model_rf_gs = RandomForestRegressor(random_state=0)

In [ ]:
param_grid = {'n_estimators': [600, 800, 1000], # 3
              'max_depth': list(range(6, 11)), # 5
              'min_samples_split': list(range(2, 5)), # 3
              'max_features': [0.90, 0.95, 1.0]} # 3

In [ ]:
gs = GridSearchCV(model_rf_gs, param_grid, cv=3, scoring='neg_mean_squared_error')
gs.fit(X_train, y_train)

https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter

In [ ]:
len(gs.cv_results_['params'])

In [ ]:
gs.cv_results_['params']

In [ ]:
gs.best_params_

In [ ]:
gs.cv_results_['mean_test_score']

In [ ]:
print(gs.score(X_val, y_val))

**RandomizedSearch**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, truncnorm, randint

In [ ]:
model_params = {
    # Muestreo aleatorio de enteros entre 600 y 1000 estimadores
    'n_estimators': randint(600, 1000),
    # Muestreo aleatorios de enteros entre profundidades máximas de 4 y 12
    'max_depth': randint(5, 12),
    # max_features con distribución normal, con media .85 y desviación estándar 0.1, acotado entre 0 y 1
    'max_features': truncnorm(a=0, b=1, loc=0.85, scale=0.1),
    # max_features con distribución uniforme entre 0.01 y 1.0 (0.01 + 0.999)
    # 'max_features': uniform(0.01, 0.999)
}

samples = 15  # number of random samples

In [ ]:
model_rf_rs = RandomForestRegressor(random_state=0)
rs = RandomizedSearchCV(model_rf_rs, param_distributions=model_params, n_iter=samples, cv=3,
                        scoring="neg_mean_squared_error")

In [ ]:
rs.fit(X_train, y_train)

In [ ]:
print(rs.best_params_)

In [ ]:
rs.cv_results_['mean_test_score']

In [ ]:
print(rs.score(X_val, y_val))

### **######### Todo #########**
Utilice XGBoost para regresion (revise la clase `XGBRegressor` https://www.geeksforgeeks.org/xgboost-for-regression/) y `RandomizedSearchCV` con 5-fold cross-validation y 100 samples para optimizar 3 hiperparámetros que considere relevantes.

Responda: ¿El mejor modelo encontrado es mejor que los resultados obtenidos con Random Forest?

In [ ]:
!pip install xgboost

**Bayesian Search**

Optuna es una librería que permite realizar búsquedas basadas en Estadística Bayesiana. https://optuna.org/#code_examples

In [ ]:
!pip install optuna

In [ ]:
import optuna

In [ ]:
def objective(trial):
    # Sugiere valores como hiperparámetros
    n_estimators = trial.suggest_int("n_estimators", 300, 800, log=True)
    max_depth = trial.suggest_int("max_depth", 6, 11)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 4)

    # Crea y ajusta un modelo
    model = RandomForestRegressor(
      n_estimators=n_estimators,
      max_depth=max_depth,
      min_samples_split=min_samples_split,
      random_state=0,
    )
    model.fit(X_train, y_train)

    # Predice y calcula el MSE
    y_pred = model.predict(X_val)
    mse = mean_squared_error(y_val, y_pred)

    # Devuelve MSE
    return mse

In [ ]:
study = optuna.create_study(direction="minimize")

In [ ]:
study.optimize(objective, n_trials=10, show_progress_bar=True)

In [ ]:
print("Best trial:", study.best_trial)
print("Best hyperparameters:", study.best_params)